# W17-D2 实验：五族 Citing 债务清算 × 定界签名盲区闭合

与 md 的分工：md 是裁决叙事（五路证据+归属论证），本 ipynb 是**可执行验证**——
① 用真门（importlib 直载 canonical_drift_ci）复算 cited 0→22 的判决翻转；
② 四迁移 DDL 直读归因 22 表；③ Context 重分布可视化；
④ G-05 v0.2 定界签名 tamper 矩阵（含首版翻车教训的防回归断言）。

In [ ]:
# matplotlib 中文字体配置（TOOLS.md 标准方式）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 实验 1：债务透镜——citing 落盘前后，同一个门的判决翻转

门的 citing 判定 = added 表名在 changes/ 目录（非下划线子目录）全文本中的**词边界命中**。
「落盘前」= 仅 g01/g0x 在册（拷贝到临时目录模拟昨天的世界）；「落盘后」= 真实 changes/（今日五案 + g0x 更新）。

In [ ]:
import importlib.util, shutil, subprocess, json, sys
from pathlib import Path

SEM = Path("/root/learning-notebooks/semantic-model")
LNKCRE = Path("/root/lnkcre")

spec = importlib.util.spec_from_file_location("canonical_drift_ci", SEM / "governance/ci/canonical_drift_ci.py")
cd = importlib.util.module_from_spec(spec); spec.loader.exec_module(cd)

def load_tables(p):
    txt = Path(p).read_text(encoding="utf-8")
    s, n = cd.load_table_set(txt)
    return s

w39 = load_tables(SEM / "governance/canonical-baseline-w39.txt")
w40 = load_tables(SEM / "governance/canonical-baseline-w40.txt")
reality = load_tables(LNKCRE / "backend/internal/platform/database/testdata/canonical_tables.txt")
added = reality - w39
print(f"w39={len(w39)}  w40={len(w40)}  reality={len(reality)}  added(w39透镜)={len(added)}")
assert added == w40 - w39, "双基线 diff 与 w39 透镜不一致"

# 落盘前：临时 changes 目录只含 g01 / g0x（模拟 2026-09-21 收盘状态）
import tempfile
with tempfile.TemporaryDirectory() as td:
    before_dir = Path(td) / "changes"; before_dir.mkdir()
    for d in ("g01-ontology-governance-header", "g0x-semantic-boundary-governance"):
        shutil.copytree(SEM / "changes" / d, before_dir / d)
    before_text, before_ids = cd.read_citing(before_dir)
after_text, after_ids = cd.read_citing(SEM / "changes")

cited_before = {t for t in added if cd.word_rx(t).search(before_text)}
cited_after  = {t for t in added if cd.word_rx(t).search(after_text)}
leaked_after = added - cited_after
print(f"在册 change：前 {before_ids} → 后 {after_ids}")
print(f"cited：{len(cited_before)} → {len(cited_after)}；leaked：{len(added)-len(cited_before)} → {len(leaked_after)}")
# 真实发现（本实验复算才暴露）：g0x What-5 族表括注「unit-pricing（含 price_authority_constraints）」
# 本身构成词边界命中——门只认文本不认意图，族名括注也是 citing。
# digest 记录的 cited 0 是取证时刻值（g0x 升格落盘之前）；09-21 收盘口径实为 cited 1。
assert cited_before == {"price_authority_constraints"}, f"落盘前口径应为 1（g0x 括注），实测 {cited_before}"
assert len(cited_after) == 22 and not leaked_after, "落盘后应 cited 22 / leaked 0"

# 真门实跑（不是复算）：w39 透镜 + w40 执勤
for lens, base in (("w39 透镜", "canonical-baseline-w39.txt"), ("w40 执勤", "canonical-baseline-w40.txt")):
    r = subprocess.run([sys.executable, str(SEM / "governance/ci/canonical_drift_ci.py"), "verify",
                        "--baseline", str(SEM / "governance" / base), "--json"],
                       capture_output=True, text=True, cwd=str(SEM))
    s = json.loads(r.stdout)["summary"]
    print(f"真门[{lens}] verdict={s['verdict']} added={s['added']} cited={s['cited']} leaked={s['leaked']} broken={s['broken']} (exit {r.returncode})")
    assert s["verdict"] == "GREEN" and r.returncode == 0
print("PASS: 债务清算 cited 0→22，真门双基线 GREEN")

In [ ]:
# 判决翻转漏斗图（昨日 vs 今日，w39 透镜口径）
fig, ax = plt.subplots(figsize=(7.5, 3.2))
cats = ["cited（有主）", "leaked（无主）"]
yesterday, today = [1, 21], [22, 0]  # 09-21 收盘含 g0x 括注隐含的 1；digest 取证时刻为 0/22
x = range(len(cats)); w = 0.35
ax.bar([i - w/2 for i in x], yesterday, w, label="09-21 落盘前", color="#c0504d")
ax.bar([i + w/2 for i in x], today,      w, label="09-22 五案落盘后", color="#4f81bd")
for i, (a, b) in enumerate(zip(yesterday, today)):
    ax.text(i - w/2, a + 0.4, str(a), ha="center"); ax.text(i + w/2, b + 0.4, str(b), ha="center")
ax.set_xticks(list(x)); ax.set_xticklabels(cats); ax.set_ylabel("表数")
ax.set_title("22 表债务清算：cited 1→22 / leaked 21→0（w39 透镜；digest 取证时刻为 0→22）")
ax.legend(); plt.tight_layout()
plt.savefig("/root/learning-notebooks/w17d2_citing_funnel.png", dpi=130); plt.show()

## 实验 2：五族×迁移归因——四份 .up.sql 直读对账

从 PG 迁移正文正则提取 CREATE TABLE，与双基线 diff 的 22 表全量对账（每表必须有出生证明）。

In [ ]:
import re
MIGS = {
    "000227": "000227_leasing_policy_unit_pricing.up.sql",
    "000229": "000229_unit_pricing_batch_model.up.sql",
    "000233": "000233_indicator_target_value_layer.up.sql",
    "000234": "000234_leasing_progress_phase1.up.sql",
}
MIG_DIR = LNKCRE / "backend/internal/platform/database/migrations-pg"
FAMILY = {
    "leasing-progress": ["leasing_stage_templates", "leasing_stage_template_items",
        "leasing_stage_completion_facts", "leasing_stage_plan_overrides",
        "leasing_progress_tasks", "leasing_progress_task_adjustments",
        "leasing_progress_settings", "leasing_progress_setting_audits",
        "leasing_progress_event_consumptions"],
    "unit_leasing": ["unit_leasing_plans", "unit_leasing_stages", "leasing_plan_recalc_batches"],
    "indicator-target": ["target_indicators", "indicator_monthly_targets", "indicator_monthly_target_history"],
    "leasing-policy": ["leasing_policies", "leasing_policy_versions", "leasing_policy_lifecycle"],
    "unit-pricing": ["price_authority_constraints", "unit_pricing_batch_ops",
        "unit_pricing_batches", "unit_pricing_batch_lines"],
}
rx = re.compile(r'CREATE TABLE (?:IF NOT EXISTS )?"?([a-z_][a-z0-9_]*)"?\s*\(', re.I)
born = {}  # table -> migration
for mid, fname in MIGS.items():
    for t in rx.findall((MIG_DIR / fname).read_text(encoding="utf-8")):
        born.setdefault(t, mid)
fam_of = {t: f for f, ts in FAMILY.items() for t in ts}

print(f"{'族':18s} {'表数':>4s}  迁移出处")
for f, ts in FAMILY.items():
    srcs = sorted({born.get(t, "?") for t in ts})
    print(f"{f:18s} {len(ts):>4d}  {', '.join(srcs)}  ({', '.join(ts)})")
assert set(born) == added, f"迁移并集 {len(born)} ≠ added {len(added)}：{set(born) ^ added}"
assert all(t in born for ts in FAMILY.values() for t in ts), "族内有表无出生证明"
assert {f: len(ts) for f, ts in FAMILY.items()} == {
    "leasing-progress": 9, "unit_leasing": 3, "indicator-target": 3,
    "leasing-policy": 3, "unit-pricing": 4}
print("PASS: 四迁移并集 == 22 added；五族 9/3/3/3/4 全部有出生证明")

## 实验 3：Context 重分布——22 表落位后的 17 Context 版图变化

裁决分布：03 Leasing Pipeline ×15 / 17 BI & Analytics ×3 / 01 Asset Foundation ×4（依据见五个 citing change）。

In [ ]:
import yaml
model = yaml.safe_load((SEM / "mi-cre-semantic-model-v0.1.yaml").read_text(encoding="utf-8"))
counts = {c: v["tables"] for c, v in model["entity"]["contexts"].items()}
DELTA = {"03 Leasing Pipeline": 15, "17 BI & Analytics": 3, "01 Asset Foundation": 4}
assert sum(DELTA.values()) == 22

show = sorted(DELTA, key=lambda c: -DELTA[c])
fig, ax = plt.subplots(figsize=(8, 3.6))
ypos = range(len(show))
ax.barh([i + 0.19 for i in ypos], [counts[c] for c in show], 0.38, label="登记前（v0.1.1）", color="#9bbb59")
ax.barh([i - 0.19 for i in ypos], [counts[c] + DELTA[c] for c in show], 0.38, label="五族入域后（v0.2 组装待合并）", color="#4f81bd")
for i, c in enumerate(show):
    ax.text(counts[c] + DELTA[c] + 0.6, i - 0.19, f"+{DELTA[c]}", va="center", color="#4f81bd")
ax.set_yticks(list(ypos)); ax.set_yticklabels(show); ax.invert_yaxis()
ax.set_xlabel("表数"); ax.set_title("W17-D2 五族 citing 的 Context 重分布（yaml 快照不动，change 先行）")
ax.legend(loc="lower right"); plt.tight_layout()
plt.savefig("/root/learning-notebooks/w17d2_context_shift.png", dpi=130); plt.show()
print("落位后头部：", {c: counts[c] + d for c, d in sorted(DELTA.items(), key=lambda kv: -(counts[kv[0]]+kv[1]))})

## 实验 4：G-05 v0.2 定界签名——扩名盲区闭合 + 首版翻车防回归

v0.2 首版两侧全边界曾把 7 个 `func Foo(` 型锚点误判 BROKEN（`(` 后随实参被当扩名）。
本实验用真 check_anchor 注入六场景：盲区必须闭合，**合法命中必须零误伤**（首版教训固化为断言）。

In [ ]:
spec2 = importlib.util.spec_from_file_location("frozen_effect_ci", SEM / "governance/ci/frozen_effect_ci.py")
fe = importlib.util.module_from_spec(spec2); spec2.loader.exec_module(fe)

def old_substring_check(repo, a):  # v0.1 逻辑复刻（子串）
    lines = (repo / a["file"]).read_text(encoding="utf-8").splitlines()
    ln, exp = int(a["line"]), a["expect"]
    if 1 <= ln <= len(lines) and exp in lines[ln - 1]: return "OK"
    return "DRIFTED" if any(exp in l for l in lines) else "BROKEN"

BASE = ["// lease states", 'StatusDraft LeaseStatus = "draft"', 'StatusActive LeaseStatus = "active"']
SCEN = {  # 对锚点 (m.go, line 2, expect 'StatusDraft') 的篡改
    "未篡改":     None,
    "行位移":     ["// new header comment"] + BASE,
    "改名":       ["// f", 'StatusX LeaseStatus = "draft"', "…"],
    "删除":       ["// f", "…"],
    "扩名(后缀)": ["// f", 'StatusDraftLegacy LeaseStatus = "draft"', "…"],
    "扩名(前缀)": ["// f", 'XStatusDraft LeaseStatus = "draft"', "…"],
}
rows = []
with tempfile.TemporaryDirectory() as td:
    for name, lines in SCEN.items():
        (Path(td) / "m.go").write_text("\n".join(lines or BASE) + "\n", encoding="utf-8")
        a = {"file": "m.go", "line": 2, "expect": "StatusDraft"}
        rows.append((name, old_substring_check(Path(td), a), fe.check_anchor(Path(td), a)[0]))
    # 首版翻车场景：`(` 右缘的合法命中不得误伤
    (Path(td) / "m.go").write_text("package occ\nfunc ValidateTransition(from, to string) error {\n\treturn nil\n}\n", encoding="utf-8")
    a = {"file": "m.go", "line": 2, "expect": "func ValidateTransition("}
    paren_old, paren_new = old_substring_check(Path(td), a), fe.check_anchor(Path(td), a)[0]
    rows.append(("`(`右缘合法命中", paren_old, paren_new))

print(f"{'场景':16s} {'v0.1子串':10s} {'v0.2定界':10s}")
for r in rows: print(f"{r[0]:16s} {r[1]:10s} {r[2]:10s}")
m = {r[0]: (r[1], r[2]) for r in rows}
assert m["未篡改"][1] == "OK", "误伤：未篡改必须 OK"
assert m["扩名(后缀)"] == ("OK", "BROKEN") and m["扩名(前缀)"] == ("OK", "BROKEN"), "盲区未闭合"
assert m["改名"][1] == m["删除"][1] == "BROKEN", "原有检出能力退化"
assert paren_new == "OK", "首版翻车回归：`(` 右缘合法命中被误判"
print("PASS: 扩名 0%→100% 检出；未篡改/`(`右缘 零误伤（7 假 BROKEN 教训已固化为断言）")

In [ ]:
# 检出率对照图：被篡改场景中门给出非 OK 判决的比例
tamper = ["改名", "删除", "扩名(后缀)", "扩名(前缀)"]
det_old = sum(m[t][0] != "OK" for t in tamper) / len(tamper)
det_new = sum(m[t][1] != "OK" for t in tamper) / len(tamper)
fig, ax = plt.subplots(figsize=(6.2, 3))
ax.bar(["v0.1 子串", "v0.2 定界签名"], [det_old * 100, det_new * 100], color=["#c0504d", "#4f81bd"], width=0.5)
for i, v in enumerate([det_old, det_new]):
    ax.text(i, v * 100 + 3, f"{v:.0%}", ha="center")
ax.set_ylabel("篡改检出率"); ax.set_ylim(0, 118)
ax.set_title("G-05 expect 升级：扩名盲区 0%→100%（且存量 15 锚零误伤）")
plt.tight_layout(); plt.savefig("/root/learning-notebooks/w17d2_tamper_matrix.png", dpi=130); plt.show()

print("== W17-D2 实验总结 ==")
print(f"① 债务清算：cited 1→{len(cited_after)}/22（digest 取证时刻 0），leaked 0，真门双基线 GREEN")
print(f"② 归因：四迁移并集 == 22 added，五族 9/3/3/3/4 出生证明齐全")
print(f"③ Context：03 +15 / 17 +3 / 01 +4（yaml 快照不动，change 先行）")
print(f"④ 定界签名：篡改检出 {det_old:.0%}→{det_new:.0%}，零误伤断言全过")